In [ ]:
import os, json, re, math
from typing import List, Dict, Any
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

# states
FSM_STATES = ["init", "deduce", "augment", "uncertain", "backtrack", "closure"]


def parse_annotation_blocks(text: str) -> List[Dict[str, str]]:
    spans = []
    # Dynamically allow augment subtypes using regex:
    # e.g., ["augment-test"], ["augment-knowledge"], etc.
    state_pattern = "|".join(FSM_STATES) + r"|augment-[a-zA-Z0-9_-]+"
    pattern = re.compile(rf'\["({state_pattern})"\]\s*(.*?)\s*\["end"\]', re.S)

    for m in pattern.finditer(text):
        raw_state = m.group(1).strip()
        seg = m.group(2).strip()

        # Normalize augment subtypes
        if raw_state.startswith("augment-"):
            state = "augment"
        else:
            state = raw_state

        if state in FSM_STATES and seg:
            spans.append({"state": state, "text": seg})

    return spans

def merge_consecutive_states(spans: List[Dict[str, str]]) -> List[Dict[str, str]]:
    """
    Merge consecutive spans with the same 'state' into a single span,
    concatenating their text with a space.
    """
    if not spans:
        return []

    merged = [spans[0].copy()]

    for s in spans[1:]:
        if s["state"] == merged[-1]["state"]:
            # Merge text if the state is the same as previous
            merged[-1]["text"] += " " + s["text"]
        else:
            # Otherwise, start a new segment
            merged.append(s.copy())

    return merged

def state_frequencies(spans):
    c = Counter([s['state'] for s in spans])
    return {s: c.get(s, 0) for s in FSM_STATES}

def transitions(spans):
    t = []
    for i in range(len(spans) - 1):
        t.append((spans[i]['state'], spans[i + 1]['state']))
    return t

def transition_matrix(list_of_spans_lists):
    counts = defaultdict(int)
    outcount = Counter()
    for spans in list_of_spans_lists:
        for a, b in transitions(spans):
            counts[(a, b)] += 1
            outcount[a] += 1
    mat = pd.DataFrame(0.0, index=FSM_STATES, columns=FSM_STATES)
    for (a, b), c in counts.items():
        mat.loc[a, b] = c / max(1, outcount[a])
    return mat

In [ ]:
def plot_heatmap(mat, title, output_path, cmap="Blues"):
    """Plot and save a transition matrix heatmap"""
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        mat,
        annot=True,
        fmt=".3f",
        cmap=cmap,
        cbar=True,
        square=True,
        linewidths=0.5,
        linecolor='white',
        vmin=-1 if cmap == "coolwarm" else 0,
        vmax=1,
        center=0 if cmap == "coolwarm" else None
    )
    plt.title(title, fontsize=14)
    plt.xlabel("To State", fontsize=12)
    plt.ylabel("From State", fontsize=12)
    plt.xticks(rotation=0, fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")

In [ ]:
def process_file(input_file, output_dir):
    """Process a single JSON file and generate all heatmaps"""
    
    # Load data
    with open(input_file, "r") as f:
        data = json.load(f)
    
    # Parse records
    records = []
    for idx, item in enumerate(data):
        annotation_text = item.get("annotation", "")
        spans = parse_annotation_blocks(annotation_text)
        merged_spans = merge_consecutive_states(spans)
        records.append({
            "id": str(item.get("id", idx)),
            "prompt": item.get("question", ""),
            "spans": spans,
            "merged_spans": merged_spans,
            "answer_correct": item.get("correct")
        })
    
    # Extract model name from filename
    filename = os.path.basename(input_file)
    model_name = filename.replace("", "").replace(".json", "") ## update here
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Separate correct and wrong answers
    span_type = 'merged_spans'
    all_spans = [tr[span_type] for tr in records if tr[span_type]]
    correct_spans = [tr[span_type] for tr in records if tr[span_type] and tr['answer_correct'] == 1]
    wrong_spans = [tr[span_type] for tr in records if tr[span_type] and tr['answer_correct'] in [0, -1]]
    
    print(f"\nProcessing: {model_name}")
    print(f"Total samples: {len(all_spans)}")
    print(f"Correct answers: {len(correct_spans)}")
    print(f"Wrong answers: {len(wrong_spans)}")
    
    # Generate transition matrices
    mat_all = transition_matrix(all_spans)
    mat_correct = transition_matrix(correct_spans) if correct_spans else pd.DataFrame(0.0, index=FSM_STATES, columns=FSM_STATES)
    mat_wrong = transition_matrix(wrong_spans) if wrong_spans else pd.DataFrame(0.0, index=FSM_STATES, columns=FSM_STATES)
    mat_diff = mat_correct - mat_wrong
    
    # Generate and save heatmaps
    base_name = os.path.join(output_dir, model_name)
    
    # 1. All transitions
    plot_heatmap(
        mat_all,
        f"{model_name}",
        f"{base_name}_all.png",
        cmap="Blues"
    )
    
    # 2. Correct transitions
    plot_heatmap(
        mat_correct,
        f"{model_name}",
        f"{base_name}_correct.png",
        cmap="Blues"
    )
    
    # 3. Wrong transitions
    plot_heatmap(
        mat_wrong,
        f"{model_name}",
        f"{base_name}_wrong.png",
        cmap="Blues"
    )
    
    # 4. Difference (Correct - Wrong)
    plot_heatmap(
        mat_diff,
        f"{model_name}",
        f"{base_name}_diff.png",
        cmap="coolwarm"
    )
    
    return mat_all, mat_correct, mat_wrong, mat_diff

In [ ]:
# Main execution
inp_files = []
output_directory = ""

# Process each file
results = {}
for inp_file in inp_files:
    input_path = inp_file
    if os.path.exists(input_path):
        matrices = process_file(input_path, output_directory)
        results[inp_file] = matrices
    else:
        print(f"File not found: {input_path}")

print(f"\nAll heatmaps saved to: {output_directory}")